# 2. K-Fold Cross-Validation

Evaluate the **XGBoost** fraud detection model using **Stratified K-Fold Cross-Validation** on the SMOTENC-resampled training data.

The optimal decision threshold of **0.70** (from Experiment 3) is applied to each fold's predicted probabilities before computing metrics.

```
Full Training Dataset (SMOTENC-resampled)
─────────────────────────────────────────────────────────────────────────────
│  Fold 1  │  Fold 2  │  Fold 3  │  Fold 4  │  Fold 5  │  Fold 6  │
─────────────────────────────────────────────────────────────────────────────
Each iteration: 5 folds = training, 1 fold = validation
```

## 1. Imports & Setup

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print('Imports complete.')

Imports complete.


## 2. Load Prepared Artifacts

In [2]:
BASE_DIR = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'artifacts').exists() or (candidate / 'requirements.txt').exists():
        BASE_DIR = candidate
        break
if BASE_DIR is None:
    BASE_DIR = Path.cwd()

ARTIFACT_DIR = BASE_DIR / 'artifacts' / 'data'
EVAL_DIR     = BASE_DIR / 'artifacts' / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_train.npz')['X_train']
X_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_X_test.npz')['X_test']
y_train = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_train.npz')['y_train']
y_test  = np.load(ARTIFACT_DIR / 'credit_card_fraud_y_test.npz')['y_test']

with open(ARTIFACT_DIR / 'features.json', 'r') as f:
    feature_names = json.load(f)

with open(ARTIFACT_DIR / 'threshold.json', 'r') as f:
    OPTIMAL_THRESHOLD = json.load(f)['optimal_threshold']

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names)

print(f'X_train : {X_train.shape}  (fraud in y_train: {int(y_train.sum()):,})')
print(f'X_test  : {X_test.shape}   (fraud in y_test:  {int(y_test.sum()):,})')
print(f'Optimal threshold: {OPTIMAL_THRESHOLD}')

X_train : (1134468, 29)  (fraud in y_train: 103,133)
X_test  : (259335, 29)   (fraud in y_test:  1,501)
Optimal threshold: 0.7


## 3. Configure Stratified K-Fold CV

Use **6-fold stratified** cross-validation so that each fold preserves the fraud class ratio.

In [3]:
N_SPLITS = 6
cv = StratifiedKFold(n_splits=N_SPLITS, random_state=SEED, shuffle=True)
print(f'Stratified K-Fold configured: {N_SPLITS} splits, random_state={SEED}')

Stratified K-Fold configured: 6 splits, random_state=42


## 4. Run Cross-Validation

Train an XGBoost model on each fold's training portion and evaluate with the **optimal threshold (0.70)** on the validation portion.

In [4]:
fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train_df, y_train), start=1):
    X_fold_train = X_train_df.iloc[train_idx]
    X_fold_val   = X_train_df.iloc[val_idx]
    y_fold_train = y_train[train_idx]
    y_fold_val   = y_train[val_idx]

    # Train XGBoost on this fold
    model = xgb.XGBClassifier(
        random_state=SEED,
        eval_metric='logloss',
        n_estimators=100,
        max_depth=6,
        n_jobs=-1,
    )
    model.fit(X_fold_train, y_fold_train)

    # Predict with optimal threshold
    y_proba_val = model.predict_proba(X_fold_val)[:, 1]
    y_pred_val  = (y_proba_val >= OPTIMAL_THRESHOLD).astype(int)

    fold_results.append({
        'fold'      : fold_idx,
        'accuracy'  : accuracy_score(y_fold_val, y_pred_val),
        'precision' : precision_score(y_fold_val, y_pred_val, zero_division=0),
        'recall'    : recall_score(y_fold_val, y_pred_val, zero_division=0),
        'f1'        : f1_score(y_fold_val, y_pred_val, zero_division=0),
        'roc_auc'   : roc_auc_score(y_fold_val, y_proba_val),
    })
    print(f"Fold {fold_idx}/{N_SPLITS}  "
          f"Prec={fold_results[-1]['precision']:.4f}  "
          f"Rec={fold_results[-1]['recall']:.4f}  "
          f"F1={fold_results[-1]['f1']:.4f}  "
          f"AUC={fold_results[-1]['roc_auc']:.4f}")

Fold 1/6  Prec=0.9906  Rec=0.9604  F1=0.9753  AUC=0.9997
Fold 2/6  Prec=0.9904  Rec=0.9640  F1=0.9770  AUC=0.9998
Fold 3/6  Prec=0.9910  Rec=0.9643  F1=0.9775  AUC=0.9998
Fold 4/6  Prec=0.9912  Rec=0.9636  F1=0.9772  AUC=0.9997
Fold 5/6  Prec=0.9898  Rec=0.9607  F1=0.9751  AUC=0.9998
Fold 6/6  Prec=0.9904  Rec=0.9610  F1=0.9755  AUC=0.9997


## 5. Cross-Validation Summary

In [5]:
results_df = pd.DataFrame(fold_results).set_index('fold')
means = results_df.mean()
stds  = results_df.std()

print('\n=== Cross-Validation Results per Fold ===')
print(results_df.round(4).to_string())

print('\n=== Mean ± Std across folds ===')
for metric in results_df.columns:
    print(f'  {metric:<12}: {means[metric]:.4f} ± {stds[metric]:.4f}')


=== Cross-Validation Results per Fold ===
      accuracy  precision  recall      f1  roc_auc
fold                                              
1       0.9956     0.9906  0.9604  0.9753   0.9997
2       0.9959     0.9904  0.9640  0.9770   0.9998
3       0.9960     0.9910  0.9643  0.9775   0.9998
4       0.9959     0.9912  0.9636  0.9772   0.9997
5       0.9955     0.9898  0.9607  0.9751   0.9998
6       0.9956     0.9904  0.9610  0.9755   0.9997

=== Mean ± Std across folds ===
  accuracy    : 0.9957 ± 0.0002
  precision   : 0.9906 ± 0.0005
  recall      : 0.9623 ± 0.0018
  f1          : 0.9763 ± 0.0011
  roc_auc     : 0.9997 ± 0.0000


## 6. Visualize CV Metric Distributions

In [6]:
metrics_to_plot = ['precision', 'recall', 'f1', 'roc_auc']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(16, 4))

for ax, metric in zip(axes, metrics_to_plot):
    values = results_df[metric].values
    ax.bar(range(1, N_SPLITS + 1), values, color='steelblue', alpha=0.8)
    ax.axhline(means[metric], color='tomato', linestyle='--', label=f'Mean={means[metric]:.4f}')
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Fold')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.suptitle(f'XGBoost — {N_SPLITS}-Fold CV (threshold={OPTIMAL_THRESHOLD})', fontsize=13)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'kfold_cv_metrics.png', dpi=150)
plt.show()
print('CV metrics chart saved.')

CV metrics chart saved.


## 7. Final Evaluation on Held-Out Test Set

Retrain on the **full** SMOTENC training split and evaluate on the untouched test set.

In [7]:
final_model = xgb.XGBClassifier(
    random_state=SEED,
    eval_metric='logloss',
    n_estimators=100,
    max_depth=6,
    n_jobs=-1,
)
final_model.fit(X_train_df, y_train)

y_proba_test = final_model.predict_proba(X_test_df)[:, 1]
y_pred_test  = (y_proba_test >= OPTIMAL_THRESHOLD).astype(int)

print('=== Final Test-Set Metrics ===')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_test):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred_test):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred_test):.4f}')
print(f'  F1-Score  : {f1_score(y_test, y_pred_test):.4f}')
print(f'  ROC AUC   : {roc_auc_score(y_test, y_proba_test):.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Legitimate', 'Fraud'],
    yticklabels=['Legitimate', 'Fraud'],
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Final Model — Confusion Matrix (threshold={OPTIMAL_THRESHOLD})')
plt.tight_layout()
plt.savefig(EVAL_DIR / 'kfold_final_confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved.')

=== Final Test-Set Metrics ===
  Accuracy  : 0.9979
  Precision : 0.8243
  Recall    : 0.8035
  F1-Score  : 0.8138
  ROC AUC   : 0.9980
Confusion matrix saved.
